# Model training 

In [55]:
import pandas as pd
import numpy as np 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import (classification_report, confusion_matrix, log_loss,
                             balanced_accuracy_score, matthews_corrcoef)

In [56]:
data = pd.read_feather("../data/less_features.feather")

In [57]:
data

,gold_ret
Date,
2018-02-01,0.002692
2018-02-02,0.003950
2018-02-05,-0.007916
2018-02-06,-0.000525
2018-02-07,-0.005190
...,...
2025-12-05,0.002478
2025-12-08,0.000000
2025-12-09,-0.005981


In [58]:
results = pd.read_feather("../data/final_data_ny_big.feather")

In [59]:
y_ny = results[["y_ny"]]
y_ny

,y_ny
Ticker,
Date,
2018-02-01,0.0
2018-02-02,0.0
2018-02-05,0.0
2018-02-06,1.0
2018-02-07,0.0
...,...
2025-12-05,1.0
2025-12-08,0.0


In [60]:
y = y_ny.iloc[:, 0].loc[data.index].astype(int)  # flatten to a Series, aligned on dates

# chronological 80/20 split (same as 3_Feature_Selection), no shuffling
split = int(len(data) * 0.8)
X_train, X_test = data.iloc[:split], data.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [61]:
# C picked by time-series CV: each fold trains on the past, validates on the next block
model = make_pipeline(
    StandardScaler(),
    LogisticRegressionCV(Cs=np.logspace(-4, 2, 30), cv=TimeSeriesSplit(n_splits=5),
                         scoring="roc_auc", max_iter=10000),
).fit(X_train, y_train)

/opt/anaconda3/envs/NASDAQ_Prediction-Project/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/opt/anaconda3/envs/NASDAQ_Prediction-Project/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnin

In [62]:
model 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregressioncv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](1,)",['gold_ret']
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,1
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [63]:
lr = model[-1]
p_test = model.predict_proba(X_test)[:, 1]
print(f"best C:   {lr.C_[0]:.4f}")
print(f"CV AUC:   {lr.scores_[1].mean(axis=0).max():.3f}")
print(f"test AUC: {roc_auc_score(y_test, p_test):.3f}")
print(f"test acc: {accuracy_score(y_test, p_test > 0.5):.3f}   vs always-up baseline {y_test.mean():.3f}")
pd.Series(lr.coef_[0], index=data.columns).sort_values()

best C:   0.0001
CV AUC:   0.537
test AUC: 0.540
test acc: 0.559   vs always-up baseline 0.559


gold_ret    0.005452
dtype: float64

## Recommendations to improve the model

**Short version:** the code is fine. No data from the future leaks in, and the overnight features stop at 09:00. The problem is *how we measure the model* and *what we ask it to predict*.

---

### 1. The CV AUC of 0.60 is too optimistic
**What happened:** in `3_Feature_Selection` we tried many feature sets, kept the best one (k=39 plus the sign filter), and then reported the score **on the same folds we used to choose it**. That is like picking the best of 100 coin-flippers and calling them skilled. The score looks good partly because we went looking for a good score.

**Evidence:** on data the selection never saw, the test AUC is **0.500**, which is pure chance. The full L1 model without the selection gets a CV AUC of about 0.52. Those are the honest numbers.

**Fix:** choose features using only the training part of each fold (nested CV), or keep a separate validation period between train and test just for making choices. Don't choose k by looking at the test AUC either, because then the test set is no longer unseen.

### 2. Remove the 92 lagged single-stock returns
With only 722 training days and around 100 features, the model will find patterns that are just luck. "ADI's return yesterday predicts the NASDAQ today" is almost certainly one of those. Use a small set of features that each have an economic reason to matter:

`ovn_ret, ovn_vol, ovn_range, vxn, vix, ndx_ret_ny_prev, ndx_ret_1d, ndx_vol_20d, asia_ret, europe_ret, prev_ny_high_dist, prev_ny_low_dist, pct_up, Dayofweek`

### 3. Change the target (the biggest improvement)
Predicting **direction** (up or down) is very hard, close to impossible. Predicting **how big the move will be** is much easier, because volatility clusters: calm days tend to follow calm days and wild days follow wild days.

Results with the 14 features above:

| What we predict | Model | CV AUC | Test AUC | Test 95% CI |
|---|---|---|---|---|
| Up or down | Logistic regression | 0.508 | 0.575 | [0.49, 0.67] |
| Up or down | Gradient boosting | 0.491 | 0.544 | [0.46, 0.64] |
| **Big move or not** (\|ret\| > median) | **Logistic regression** | **0.613** | **0.660** | **[0.56, 0.74]** |

Only the "big move" model has a confidence interval that stays above 0.5, so it is the only result we can show is better than chance.

Options:
- predict the size of the move (`|ret_ny|` or the intraday range) with a regression
- keep predicting direction, but drop days with tiny moves (those labels are basically random)
- two steps: first predict whether the move will be big, then predict direction only on big-move days

### 4. More data
The test set has only 144 days, so any test AUC is uncertain by about ±0.09. With that much uncertainty, 0.50 and 0.57 can't be told apart. The NQ overnight data only starts in 2023, but the daily yfinance features (VIX, yields, NDX lags) go back about 20 years, which is roughly 5,000 days. Train a model on daily features only using that long history, then check whether adding the overnight features improves it on 2023 onward.

### 5. Report results more honestly
- **Report a confidence interval** for the test AUC (bootstrap the test days), so readers can see how much noise there is.
- **Compare log loss, not accuracy.** The market went up 57% of the time, so "always predict up" already scores 57% accuracy. Our 53% is worse than that. Log loss compared with simply predicting the base rate is a fairer test.
- **Walk-forward testing:** retrain every month on all past data and predict the next month, instead of a single 80/20 split. This is closer to how the model would actually be used.

---

**Priority:** do **1** and **3**. "We can predict volatility but not direction, and here is properly validated evidence" is a stronger project than chasing an AUC of 0.52 on direction.
